In [ ]:
import pandas as pd

routes = pd.read_csv("gtfs/routes.txt", dtype=str)
trips = pd.read_csv("gtfs/trips.txt", dtype=str)
stops = pd.read_csv("gtfs/stops.txt", dtype=str)

routes.head()


In [ ]:
metro_routes = routes[routes["route_type"] == "1"]
metro_routes

In [ ]:
metro_trips = trips[trips["route_id"].isin(metro_routes["route_id"])]
len(metro_trips)

In [ ]:
metro_trips.head()

In [ ]:
metro_trip_ids = set(metro_trips["trip_id"])

chunks = pd.read_csv("gtfs/stop_times.txt", dtype=str, chunksize=500_000)
metro_stop_times = pd.concat(
    chunk[chunk["trip_id"].isin(metro_trip_ids)] for chunk in chunks
)

len(metro_stop_times)

In [ ]:
# Pick the first Orange line trip (route_id "2")
orange_trip = metro_trips[metro_trips["route_id"] == "2"].iloc[0]
print(orange_trip["trip_id"], "→", orange_trip["trip_headsign"])

# Get its stops, in order
one_trip = metro_stop_times[metro_stop_times["trip_id"] == orange_trip["trip_id"]].copy()
one_trip["stop_sequence"] = one_trip["stop_sequence"].astype(int)
one_trip = one_trip.sort_values("stop_sequence")

# Attach station names
one_trip = one_trip.merge(stops[["stop_id", "stop_name"]], on="stop_id")

one_trip[["stop_sequence", "stop_name", "arrival_time", "departure_time"]]

In [ ]:
def to_seconds(t):
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)

one_trip["arr_sec"] = one_trip["arrival_time"].apply(to_seconds)
one_trip["travel_to_next"] = one_trip["arr_sec"].shift(-1) - one_trip["arr_sec"]

one_trip[["stop_sequence", "stop_name", "arrival_time", "arr_sec", "travel_to_next"]]

In [ ]:
total = one_trip["arr_sec"].iloc[-1] - one_trip["arr_sec"].iloc[0]
print("Stations:", len(one_trip))
print("Total trip:", total / 60, "minutes")

one_trip.sort_values("travel_to_next", ascending=False).head(3)[["stop_name", "travel_to_next"]]

In [ ]:
trip_stops = stops[stops["stop_id"].isin(one_trip["stop_id"])]
trip_stops.head()

In [ ]:
stops[stops["stop_name"].str.contains("Jean-Talon")]

In [ ]:
import folium

# All platforms used by any metro trip, with their line
used = metro_stop_times[["trip_id", "stop_id"]].drop_duplicates("stop_id")
used = used.merge(metro_trips[["trip_id", "route_id"]], on="trip_id")
used = used.merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
used = used.merge(metro_routes[["route_id", "route_color"]], on="route_id")

m = folium.Map(location=[45.51, -73.60], zoom_start=11)

for _, row in used.iterrows():
    folium.CircleMarker(
        location=[float(row["stop_lat"]), float(row["stop_lon"])],
        radius=5,
        color="#" + row["route_color"],
        fill=True,
        tooltip=row["stop_name"],
    ).add_to(m)

m

In [ ]:
rows = []
for route_id in ["1", "2", "4", "5"]:
    trip = metro_trips[(metro_trips["route_id"] == route_id) & (metro_trips["direction_id"] == "0")].iloc[0]
    st = metro_stop_times[metro_stop_times["trip_id"] == trip["trip_id"]].copy()
    st["stop_sequence"] = st["stop_sequence"].astype(int)
    st = st.sort_values("stop_sequence").merge(stops[["stop_id", "stop_name"]], on="stop_id")
    for _, r in st.iterrows():
        rows.append({"route_id": route_id, "seq": r["stop_sequence"], "stop_name": r["stop_name"], "x": "", "y": ""})

layout = pd.DataFrame(rows)
layout.to_csv("stations_schematic.csv", index=False)
layout.groupby("route_id").size()

In [ ]:
import math

WIDTH, HEIGHT, PAD = 1000, 1000, 80
ROTATE_DEG = 0
MIN_GAP = 55      # target minimum pixels between any two stations
PASSES = 60         # relaxation rounds

# 1. One lat/lon per station name (platforms of the same station get averaged)
coords = stops[stops["stop_id"].isin(metro_stop_times["stop_id"])][["stop_name", "stop_lat", "stop_lon"]].copy()
coords[["stop_lat", "stop_lon"]] = coords[["stop_lat", "stop_lon"]].astype(float)
coords = coords.groupby("stop_name", as_index=False).mean()

layout = pd.read_csv("stations_schematic.csv", dtype=str)[["route_id", "seq", "stop_name"]]
layout = layout.merge(coords, on="stop_name", how="left")
print(layout["stop_lat"].isna().sum(), "stations missing coordinates")

# 2. Lat/lon -> flat x/y (longitude degrees shrink away from the equator)
lat0 = layout["stop_lat"].mean()
lon0 = layout["stop_lon"].mean()
x = (layout["stop_lon"] - lon0) * math.cos(math.radians(lat0))
y = -(layout["stop_lat"] - lat0)

a = math.radians(ROTATE_DEG)
x, y = x * math.cos(a) - y * math.sin(a), x * math.sin(a) + y * math.cos(a)

# 3. Scale to the canvas first, so  is in pixels
scale = min((WIDTH - 2 * PAD) / (x.max() - x.min()), (HEIGHT - 2 * PAD) / (y.max() - y.min()))
layout["x"] = (x - x.min()) * scale + PAD
layout["y"] = (y - y.min()) * scale + PAD

# 4. Push crowded stations apart, keeping the overall shape
#    Each pass nudges any pair closer than MIN_GAP away from each other.
uniq = layout.drop_duplicates("stop_name").set_index("stop_name")[["x", "y"]].copy()
names = uniq.index.tolist()
px = uniq["x"].to_numpy(dtype=float)
py = uniq["y"].to_numpy(dtype=float)

for _ in range(PASSES):
    moved = 0
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            dx, dy = px[j] - px[i], py[j] - py[i]
            d = math.hypot(dx, dy)
            if d >= MIN_GAP:
                continue
            if d < 1e-6:            # identical points: nudge arbitrarily
                dx, dy, d = 1.0, 0.0, 1.0
            push = (MIN_GAP - d) / 2
            ux, uy = dx / d, dy / d
            px = uniq["x"].to_numpy(dtype=float, copy=True)
            py = uniq["y"].to_numpy(dtype=float, copy=True)
            px[j] += ux * push
            py[j] += uy * push
            moved += 1
    if moved == 0:
        break

spread = dict(zip(names, zip(px, py)))
layout["x"] = [spread[n][0] for n in layout["stop_name"]]
layout["y"] = [spread[n][1] for n in layout["stop_name"]]

# 5. Rescale to fit the canvas again, since spreading grew the map
for col, size in (("x", WIDTH), ("y", HEIGHT)):
    lo, hi = layout[col].min(), layout[col].max()
    layout[col] = (layout[col] - lo) / (hi - lo) * (size - 2 * PAD) + PAD
layout["x"] = layout["x"].round().astype(int)
layout["y"] = layout["y"].round().astype(int)

layout[["route_id", "seq", "stop_name", "x", "y"]].to_csv("stations_schematic.csv", index=False)
print("done; min gap target:", MIN_GAP)

In [ ]:
import json, datetime

def clean_name(n):
    return n.replace("Station ", "").replace(" -Zone B", "").strip()

today = datetime.date.today()
ymd = today.strftime("%Y%m%d")
weekday = today.strftime("%A").lower()

cal = pd.read_csv("gtfs/calendar.txt", dtype=str)
active = set(cal[(cal[weekday] == "1") & (cal["start_date"] <= ymd) & (cal["end_date"] >= ymd)]["service_id"])

# Exceptions: 1 = service added on this date, 2 = service removed
cd = pd.read_csv("gtfs/calendar_dates.txt", dtype=str)
cd_today = cd[cd["date"] == ymd]
active |= set(cd_today[cd_today["exception_type"] == "1"]["service_id"])
active -= set(cd_today[cd_today["exception_type"] == "2"]["service_id"])

today_trips = metro_trips[metro_trips["service_id"].isin(active)]

st = metro_stop_times[metro_stop_times["trip_id"].isin(today_trips["trip_id"])].copy()
st["stop_sequence"] = st["stop_sequence"].astype(int)
st["t"] = st["arrival_time"].apply(to_seconds)
st = st.merge(stops[["stop_id", "stop_name"]], on="stop_id")
st = st.merge(today_trips[["trip_id", "route_id"]], on="trip_id")
st = st.sort_values(["trip_id", "stop_sequence"])

out = []
for trip_id, g in st.groupby("trip_id"):
    out.append({
        "route": g["route_id"].iloc[0],
        "stops": [clean_name(n) for n in g["stop_name"]],
        "times": g["t"].tolist(),
    })

with open("trips_today.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False)

print(len(active), "active services,", len(out), "metro trips today")

In [ ]:
# Build a network model: stations, lines, and travel times between neighbours
from collections import defaultdict

# 1. Median travel time between each pair of adjacent stations, per line
seg_times = defaultdict(list)
for trip_id, g in st.groupby("trip_id"):
    route = g["route_id"].iloc[0]
    names = [clean_name(n) for n in g["stop_name"]]
    times = g["t"].tolist()
    for i in range(len(names) - 1):
        seg_times[(route, names[i], names[i + 1])].append(times[i + 1] - times[i])

edges = []
for (route, a, b), samples in seg_times.items():
    samples.sort()
    median = samples[len(samples) // 2]
    edges.append({"route": route, "from": a, "to": b, "seconds": int(median)})

# 2. Stations: position, lines served, terminals, STM stop code
layout = pd.read_csv("stations_schematic.csv", dtype=str)
routes_by_station = defaultdict(set)
for e in edges:
    routes_by_station[e["from"]].add(e["route"])
    routes_by_station[e["to"]].add(e["route"])

terminals = set()
for trip_id, g in st.groupby("trip_id"):
    terminals.add(clean_name(g["stop_name"].iloc[0]))
    terminals.add(clean_name(g["stop_name"].iloc[-1]))

# Stop codes let us match STM's service alerts to stations
codes = stops[stops["stop_id"].isin(metro_stop_times["stop_id"])][["stop_name", "stop_code"]]
codes = {clean_name(r["stop_name"]): r["stop_code"] for _, r in codes.iterrows()}

stations = {}
for _, r in layout.drop_duplicates("stop_name").iterrows():
    name = clean_name(r["stop_name"])
    stations[name] = {
        "x": int(r["x"]),
        "y": int(r["y"]),
        "routes": sorted(routes_by_station[name]),
        "terminal": name in terminals,
        "code": codes.get(name, ""),
    }

network = {
    "routes": {r["route_id"]: {"name": r["route_long_name"], "color": "#" + r["route_color"]}
               for _, r in metro_routes.iterrows()},
    "stations": stations,
    "edges": edges,
}

with open("network.json", "w", encoding="utf-8") as f:
    json.dump(network, f, ensure_ascii=False, indent=1)

interchanges = [n for n, s in stations.items() if len(s["routes"]) > 1]
print(len(stations), "stations,", len(edges), "edges")
print("Interchanges:", sorted(interchanges))
print("Terminals:", len(terminals))
print("Berri-UQAM:", stations["Berri-UQAM"])

In [ ]:
# Crop the map to the area the stations actually occupy
xs = [s["x"] for s in stations.values()]
ys = [s["y"] for s in stations.values()]
pad = 45
box = [min(xs) - pad, min(ys) - pad, max(xs) - min(xs) + 2 * pad, max(ys) - min(ys) + 2 * pad]

network["viewBox"] = " ".join(str(int(v)) for v in box)

with open("network.json", "w", encoding="utf-8") as f:
    json.dump(network, f, ensure_ascii=False, indent=1)

print("viewBox:", network["viewBox"])